# Layer 2, P1: friendly noise that keeps head T's prediction but DISTURBS the shared features

A variant of FRIENDS (not the paper's method). FRIENDS makes eta as large as possible in pixels while head T's output stays the same.
P1 instead makes eta change the **shared 1024-number embedding** (what the old heads read) as much as possible, while head T's prediction stays the same:

    minimise  KL(head T with eta || head T without eta)  -  beta * (relative change of the shared features)

Same optimizer, cap zeta, random noise and warm-up as FRIENDS. Code: `layer2_defense/friends_p1.py` and `friends_layer2_p1.py`
(copies; the FRIENDS code is unchanged). **beta** sets how hard the features are pushed: too large and head T's prediction changes too,
which breaks P1's condition. **Read the "P1 check" table: head-T agreement should stay at ~95% or more.**

Arms (eps 0.3, seed 0, zeta = mu = 32, the setting where option (b) gained most): a beta sweep at warm-up 2 and two at warm-up 0, plus the cost on clean data.
They share the eps-0.3 results folder with notebooks 2-4, so `A_clean` / `B_poison` are reused. About 1 to 1.5 hours on a T4.

## 1. Settings

In [ ]:
EPS = '0.3'                       # '0.3' = the main table (cautious, eps 0.3); '0.1' = the weaker attack
NOISE = {
    '0.3': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.3/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.3__delta_0.3_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_29.pkl',
    '0.1': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.1/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.1__delta_0.1_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_39.pkl',
}
NOISE_PKL = NOISE[EPS]
# One folder per eps, SHARED by the option a / b / c notebooks: A_clean and B_poison then run only once, and each
# notebook's table shows every arm of that eps. Finished arms (a finished log in this folder) are skipped.
RESULTS_DIR = f'/content/drive/MyDrive/brainwash/layer2_runs/layer2_defense_eps{EPS}'
SEED = 0                            # only seed 0: there are no victim checkpoints for other seeds yet

REPO_URL = 'https://github.com/anushkathagle/brainwash-repro.git'
REPO_BRANCH = 'layer2/Friendly'

## 2. GPU and Google Drive

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isfile(NOISE_PKL), f'not found: {NOISE_PKL}'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('noise pkl:', os.path.basename(NOISE_PKL), f'({os.path.getsize(NOISE_PKL)/1e6:.0f} MB)')
print('results  :', RESULTS_DIR)

## 3. Code: the `layer2/Friendly` branch

The Layer-2 code lives in the repo, in `layer2_defense/` (next to `approaches/` and `repro/`), so nothing is written here.
The first run clones the branch; re-running this cell later **pulls the newest commit** of the branch.
Private repo: add a read-only GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar).

In [ ]:
import subprocess, sys
REPO_DIR = '/content/brainwash-repro'
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
    if tok:
        url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', '-b', REPO_BRANCH, url, REPO_DIR], check=True)
else:   # already cloned in this Colab session: move to the newest commit of the branch
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', url, REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', '-B', REPO_BRANCH, 'FETCH_HEAD'], check=True)
%cd /content/brainwash-repro
assert os.path.isfile('layer2_defense/friends_layer2.py'), 'layer2_defense/ missing: is REPO_BRANCH right?'
sys.path.insert(0, '/content/brainwash-repro/layer2_defense')
import importlib, colab_helpers as H
importlib.reload(H)
H.record_code_version(RESULTS_DIR)      # which commit produced the runs in this folder -> code_version.txt

## 4. Data and checkpoints (CIFAR-100 and the authors' victim checkpoints, about 200 MB)

In [ ]:
!pip -q install huggingface_hub
!python repro/download_artifacts.py --what cifar100
!python repro/prepare_cifar100.py

## 5. Arms

In [ ]:
def p1(beta, w=2, z=32):              # P1 noise: keep head T, disturb the shared features; z = zeta = mu (x/255)
    return f'--addnoise --friends --zeta {z} --mu_rand {z} --warmup_epochs {w} --p1_beta {beta}'

ARMS = [
    ('A_clean',          ''),                       # shared with notebooks 2-4 (skipped if already done)
    ('B_poison',         '--addnoise'),
    ('p1_w2_z32_b1',     p1(1)),
    ('p1_w2_z32_b3',     p1(3)),
    ('p1_w2_z32_b10',    p1(10)),
    ('p1_w2_z32_b0.3',   p1(0.3)),
    ('p1_w0_z32_b1',     p1(1, w=0)),
    ('p1_w0_z32_b10',    p1(10, w=0)),
    ('p1clean_w2_z32_b3', p1(3).replace('--addnoise ', '')),   # cost on clean data
]
CMD = H.base_cmd(NOISE_PKL, RESULTS_DIR, SEED).replace('layer2_defense/friends_layer2.py', 'layer2_defense/friends_layer2_p1.py')
H.run_arms(ARMS, CMD, RESULTS_DIR, SEED)

## P1 check: did eta keep head T's prediction, and how much did it move the features?

In [ ]:
import re, glob, pandas as pd
rows = []
for log in sorted(glob.glob(f'{RESULTS_DIR}/p1*_s{SEED}.log')):
    m = re.search(r'beta=([0-9.]+).*?mean\|eta\|/zeta=([0-9.]+).*?at-bound=([0-9.]+)%.*?head-T agreement=([0-9.]+)%.*?feature change=([0-9.]+)', open(log).read())
    if m:
        rows.append({'arm': os.path.basename(log)[:-len(f'_s{SEED}.log')], 'beta': float(m[1]), 'mean|eta|/zeta': float(m[2]),
                     'at cap %': float(m[3]), 'head-T agreement %': float(m[4]), 'feature change': float(m[5])})
pd.DataFrame(rows)

## Results

In % (as in the BrainWash paper). More negative BWT = more forgetting.
`recovered_%` = how much of the BWT gap between the attack (`ours`) and the clean run an arm closes (100 = as good as clean, negative = worse than no defense).
`last_vs_attack` = last-task accuracy minus the attack's: a negative number is what the recovery costs on the new task.
`old_acc_at_eta` = old-task accuracy when the friendly noise was computed (62.0 = nothing forgotten yet).

In [ ]:
H.results_table(RESULTS_DIR, SEED)

## Per-epoch old-task accuracy (rows = epochs, columns = arms): when does the damage happen?

In [ ]:
H.per_epoch_table(RESULTS_DIR, SEED)